# Automated Customer Reviews
### Goal:
The goal is to create a NLP powered web application that aggregates and analyzes customer reviews.

### Main Functionality:
1. **Classification**: Classify reviews into categories such as positive, negative, and neutral.
    - Use a model if i feel like it, but I can also use a simple keyword-based approach for a baseline.
        - if model, then use pre-trained transformer-based model

2. **Product Category Clustering**: Simplify dataset by clustering product categories into broader groups.
    - Use a clustering algorithm like K-means or hierarchical clustering on product features.

3. **Review Summarization using LLMs**: Summarize reviews into articles that recoment top products in each category.
    - Use a pre-trained LLM (T5, Bart, or Groq) for summarisation.
    - Find top complaints and worst features for each product category and worst individual products.
    - Can also generate sentiment-based summaries and sentiment analysis reports as bonus features.


In [1]:
# imports
import pandas as pd
import os

# Classification:
0. **Data Loading**: Load the dataset of Amazon product reviews.
1. **Data Cleaning and Preprocessing**: Handle missing values, clean review text, and perform tokenization.
2. **Feature Extraction**: Convert text data into numerical features using techniques like TF-IDF or word embeddings.

## Data Source:
- **Local Dataset**: Kaggle dataset of Amazon product reviews (small subset for testing)

In [2]:
if not os.path.exists("data"):
    import kagglehub, shutil
    shutil.move(
        kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products"), 
        os.path.join(os.getcwd(), "data"))

In [3]:
curr_dir = os.getcwd()

revs_df = pd.read_csv(os.path.join(curr_dir, "data", "1429_1.csv"))

/tmp/ipykernel_5065/3146233023.py:3: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  revs_df = pd.read_csv(os.path.join(curr_dir, "data", "1429_1.csv"))


## Data Cleaning and Preprocessing:
1. **Handling Missing Values**: Remove or impute missing values in the dataset.
2. **Text Preprocessing**: Clean review text by removing special characters, stop words, and performing tokenization.

In [4]:
useless_cols = [
    'id',
    'reviews.id', # not useful in our case
    'reviews.username', # might be useful for user-level analysis later
    'reviews.userCity', # might be useful for geographic analysis later
    'reviews.userProvince', # ^
    'manufacturer', # useless cuz all products are from the same manufacturer
    'reviews.sourceURLs',
    'keys',
    'asins', # not useful in our case
    'reviews.dateAdded',
    'reviews.dateSeen',
    'reviews.date', 
    'reviews.didPurchase', # useless cuz only one value is True
]

In [5]:
def clean_data(df:pd.DataFrame, drop_cols:list=[]) -> pd.DataFrame:
    # Drops columns specified
    df_clean = df.drop(columns=drop_cols, errors="ignore").dropna(subset=["reviews.text", "reviews.rating"])

    # Drops rows that have NaN in these cols
    df_clean.dropna(subset=["name", "reviews.doRecommend"], inplace=True)

    # Replaces NaN values with False then
    df_clean.replace(pd.NA, False, inplace=True)

    # Make pd info work better
    df_clean["reviews.numHelpful"] = df_clean["reviews.numHelpful"].fillna(0).astype(int)
    df_clean["reviews.rating"] = df_clean["reviews.rating"].fillna(0).astype(int)
    df_clean["reviews.doRecommend"] = df_clean["reviews.doRecommend"].fillna(False).astype(bool)

    return df_clean

revs_df_clean = clean_data(revs_df, useless_cols)

In [11]:
revs_df_clean.sample(10)

,name,brand,categories,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.text,reviews.title
25989,"Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",True,0,5,Excellent speakers and great voice control... ...,Great product
2146,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",True,0,4,The size and price makes this a best buy! I'm ...,Great product
2809,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",True,0,5,I had the original Kindle and just decided to ...,Good reader
23349,"Echo (White),,,\r\nEcho (White),,,",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",True,0,5,I really enjoy using the echo to learn and cre...,Would highly recommend the Echo
8210,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",True,0,5,We purchased this for our son (9) and it works...,Good basic tablet
1712,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",False,0,3,works great for the price.but it shoves amazon...,soso
8163,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",True,0,4,I like it kids like it definitely needs a case...,It's great
25297,Amazon - Amazon Tap Portable Bluetooth and Wi-...,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",True,0,5,"With some technical savvy, you can quickly hav...",Alexia my new best friend
24605,"Echo (White),,,\r\nEcho (White),,,",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",True,0,4,Works great and am enjoying it daily. I would ...,Love the Echo
17079,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T...",True,0,5,I bought these for my five and six-year-olds a...,Great buy! Kids Love it!


### Classification Model: